In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    PrecisionRecallDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline

from leadgate.pipeline import make_champion_pipeline, make_cv, make_preprocessor

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

In [ ]:
ct = make_preprocessor()

In [ ]:
dummy_pipe = Pipeline(
    [("preprocessor", ct), ("model", DummyClassifier(strategy="most_frequent"))]
)
logreg_pipe = make_champion_pipeline(class_weight="balanced")

In [ ]:
cv = make_cv()
dummy_cv = cross_val_score(
    dummy_pipe, X_train, y_train, cv=cv, scoring="average_precision"
)
log_cv = cross_val_score(
    logreg_pipe, X_train, y_train, cv=cv, scoring="average_precision"
)
print(
    f"mean of dummy classifier: {np.mean(dummy_cv)}, standard deviation: {np.std(dummy_cv)}"
)
print(
    f"mean of logistic regression: {np.mean(log_cv)}, standard deviation: {np.std(log_cv)}"
)

In [ ]:
y_proba = cross_val_predict(
    logreg_pipe, X_train, y_train, cv=cv, method="predict_proba"
)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)
print(classification_report(y_train, y_pred))

In [ ]:
confusion_matrix(y_train, y_pred)

In [ ]:
print(
    f"out-of-fold PR-AUC (average precision): {average_precision_score(y_train, y_proba)}"
)
PrecisionRecallDisplay.from_predictions(y_train, y_proba)
plt.title("PR-AUC curve and AP")
plt.savefig("../reports/figures/cv-PR-AUC.png")

Dummy gives 0.117 PR-AUC and logistic regression 0.400 on CV, so it is about 3x better than guessing. Out-of-fold at 0.5 it catches 63% of the yes with 27% precision. This is my floor, next models have to beat it. Test stays untouched until 07.